# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1755522541142_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-107-187.ap-southeast-1.compute.internal:39867
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1755522541142_0002/container_1755522541142_0002_01_000001/hail-20250820-0001-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [3]:
# gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
# vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'
gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/goypav/SG10K_Health/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3a://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-08-20 00:01:36.647 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [5]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
## Test
###
gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand
# gvcf_paths=ls_gvcf[0:100] # variant_data: 37755085 rows and 100 columns in 2586 partitions ~ 30Gb ~ 30min on 500 CPU onDemand
gvcf_paths

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6377/64a24a8b-25fd-42a8-93f2-22ebfd581706/output/try-1/WHB6377.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6378/e272cfc6-81ea-49b4-8694-4413c7907e9f/output/try-1/WHB6378.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6380/c5169919-0140-411e-bf2b-765553f35f1f/output/try-1/WHB6380.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6381/f2a39ab6-0342-4980-8b31-791a41f4e549/output/try-1/WHB6381.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6300/fdd1b13b-7452-45c0-954a-80560830e731/output/try-1/WHB6300.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6385/2af836d8-43b0-442f-ba4d-6f463f24c74e/output/try-1/WHB6385.hard-filtered.gvcf.gz', 's3a:

# Combine VDS

In [ ]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_combined_batch1_2_3_4_batch5_6_7_8.bf2-tr500k-sp1k.n8000.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_combined_batch9_10_11.bf2-tr500k-sp1k.n4000.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_combined_all_batches.bf2-tr500k-sp1k.n10322.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Check VDS

In [ ]:
# source
vds_prefix = 's3://precise-scratch/goypav/SG10K_Health/VDS/'

# input
vds_uri = vds_prefix + 'SG10K_Health_combined_batch1_2_3_4_batch5_6_7_8.bf2-tr500k-sp1k.n8000.vds'

In [ ]:
# read VDS
vds = hl.vds.read_vds(vds_uri)

In [ ]:
# check reference_data
vds.reference_data.describe()

In [ ]:
# check variant_data
vds.variant_data.describe()

In [ ]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

In [ ]:
hl.eval(vds.reference_data.ref_block_max_length)